# 02 — RAG Pipeline with LlamaIndex

**Goal**: Build a complete RAG system that answers questions from your own documents.

RAG = Retrieval Augmented Generation: retrieve relevant docs → feed to LLM → generate answer.

In [ ]:
from llama_index.core import (
    VectorStoreIndex, SimpleDirectoryReader, Settings,
    StorageContext, load_index_from_storage
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
import os

# Configure global settings
Settings.llm = Ollama(model="llama3.2", request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)

print("Settings configured")

## 1. Load Documents from a Directory

Let's create a small knowledge base about GenAI concepts.

In [ ]:
# Create sample documents
os.makedirs("./data/genai", exist_ok=True)

files = {
    "transformer.txt": """
The Transformer architecture, introduced in "Attention is All You Need" (2017), revolutionized NLP.
It uses self-attention mechanisms to process all tokens in parallel, unlike RNNs which process sequentially.
Key components include multi-head attention, positional encoding, and feed-forward networks.
Transformers are the foundation of all modern LLMs like GPT, Llama, and Mistral.
""",
    "rag.txt": """
Retrieval Augmented Generation (RAG) combines information retrieval with text generation.
When a user asks a question, RAG first searches a knowledge base for relevant documents.
These documents are then fed to an LLM as context to generate a grounded answer.
RAG reduces hallucinations and allows the model to access up-to-date information.
""",
    "fine_tuning.txt": """
Fine-tuning adapts a pretrained language model to a specific task or domain.
Full fine-tuning updates all parameters, while Parameter-Efficient Fine-Tuning (PEFT)
methods like LoRA only train small adapter modules. LoRA adds rank-decomposition matrices
to attention layers, reducing trainable parameters by 99% while maintaining quality.
""",
    "vectordb.txt": """
Vector databases store embeddings (vector representations of data) and enable similarity search.
They use Approximate Nearest Neighbor (ANN) algorithms like HNSW and IVF for fast retrieval.
Popular options include ChromaDB (open-source, Python-native), Pinecone (managed), and Qdrant.
Vector DBs are essential for production RAG systems that need to search millions of documents.
""",
    "prompt_engineering.txt": """
Prompt engineering is the practice of designing inputs to guide LLM outputs.
Key techniques include: few-shot prompting (providing examples), chain-of-thought (step-by-step reasoning),
and structured output (JSON mode). Good prompt engineering can dramatically improve model performance
without any model modification.
""",
}

for filename, content in files.items():
    with open(f"./data/genai/{filename}", "w") as f:
        f.write(content.strip())

print(f"Created {len(files)} documents in ./data/genai/")

## 2. Index the Documents

In [ ]:
documents = SimpleDirectoryReader("./data/genai").load_data()
print(f"Loaded {len(documents)} documents")

index = VectorStoreIndex.from_documents(documents)
print("Index created!")

## 3. Query the RAG System

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=2)

questions = [
    "What is RAG and how does it reduce hallucinations?",
    "How does the Transformer architecture work?",
    "What is LoRA and why is it useful?",
    "Name some popular vector databases."
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print('='*60)
    response = query_engine.query(q)
    print(f"A: {response}")

## 4. Persisting the Index (Save to Disk)

You don't want to re-index every time you restart.

In [ ]:
# Save
index.storage_context.persist(persist_dir="./storage")
print("Index saved to ./storage/")

# Load (in a new session)
# from llama_index.core import StorageContext, load_index_from_storage
# storage_context = StorageContext.from_defaults(persist_dir="./storage")
# loaded_index = load_index_from_storage(storage_context)
# query_engine = loaded_index.as_query_engine()
# print("Index loaded from disk")

## 5. Customizing the Prompt

You can customize how the LLM synthesizes the answer.

In [ ]:
from llama_index.core import PromptTemplate

custom_prompt = PromptTemplate(
    """You are a GenAI tutor. Answer the question using ONLY the context below.
If you don't know, say "I don't have enough information."
Be concise and educational.

Context:
{context_str}

Question: {query_str}
Answer:"""
)

query_engine = index.as_query_engine(
    text_qa_template=custom_prompt,
    similarity_top_k=2
)

response = query_engine.query("What is prompt engineering?")
print(response)

## Key Takeaways

1. **RAG pipeline**: Load docs → Index → Query Engine
2. **SimpleDirectoryReader** loads all files from a folder
3. **Persist** the index to avoid re-indexing
4. **Custom prompts** give you control over answer style
5. This is the same pattern used in production RAG systems

**Next**: Advanced LlamaIndex features (routing, sub-queries) →